# Notebook 02 - Load & Prepare Data
### Spain Tactical Analysis: Qatar 2022 - Euro 2024

**Purpose:** Extract all match events, lineups, and 360 freeze-frame data for Spain's matches from both tournaments. Consolidate them into unified raw datasets and save them locally so we never have to re-query the StatsBomb API.

**Outputs:**
- `outputs/data/events_raw.parquet`
- `outputs/data/lineups_raw.parquet`
- `outputs/data/frames_360_raw.parquet` (if 360 data exists)

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import time  # Added to prevent GitHub API rate limiting
from statsbombpy import sb
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path so we can import utils
sys.path.insert(0, os.path.abspath('..'))
from utils.config import OUTPUTS_DATA_DIR, TOURNAMENT_LABELS

print("Imports OK.")
print(f"Data directory: {OUTPUTS_DATA_DIR}")

---
## Section 1 - Load Match Inventory

In [ ]:
inventory_path = os.path.join(OUTPUTS_DATA_DIR, 'spain_match_inventory.csv')
inventory = pd.read_csv(inventory_path)

print(f"Loaded {len(inventory)} matches from inventory.")
display(inventory[['match_id', 'match_date', 'opponent', 'spain_score', 'opp_score', 'result', 'tournament']])

---
## Section 2 - Pull All Event Data

We iterate through every match in our inventory and pull the full event data from StatsBomb. 
*(Note: We use `time.sleep(1.5)` between calls to prevent GitHub from throwing a 429 Too Many Requests error)*

In [ ]:
events_list = []

# Detect which column has the stage info
stage_col = 'competition_stage' if 'competition_stage' in inventory.columns else 'stage'

print("Pulling event data...\n")
for idx, row in inventory.iterrows():
    match_id = int(row['match_id'])
    tourney  = row['tournament']
    opponent = row['opponent']
    stage    = row.get(stage_col, 'Unknown')
    
    print(f"  [{idx+1}/{len(inventory)}] Match {match_id} | {tourney} vs {opponent} ({stage})")
    
    try:
        match_events = sb.events(match_id=match_id)
        match_events['match_id']          = match_id
        match_events['tournament']        = tourney
        match_events['opponent']          = opponent
        match_events['competition_stage'] = stage
        events_list.append(match_events)
    except Exception as e:
        print(f"    [ERR] Failed: {e}")
        
    time.sleep(1.5)  # Throttle to avoid 429 Rate Limit

events_raw = pd.concat(events_list, ignore_index=True)
print(f"\nDone. Total events pulled: {len(events_raw):,}")

---
## Section 3 - Pull Lineup Data

Lineup data gives us player names, jersey numbers, and positional context for both teams in every match.

In [ ]:
lineups_list = []

print("Pulling lineup data...")
for idx, row in inventory.iterrows():
    match_id = int(row['match_id'])
    tourney  = row['tournament']
    
    try:
        match_lineups = sb.lineups(match_id=match_id)  # returns dict of DataFrames
        for team_name, df in match_lineups.items():
            df = df.copy()
            df['match_id']  = match_id
            df['tournament'] = tourney
            df['team_name'] = team_name
            lineups_list.append(df)
    except Exception as e:
        print(f"  [ERR] Lineups for match {match_id}: {e}")
        
    time.sleep(1.5)  # Throttle to avoid 429 Rate Limit

lineups_raw = pd.concat(lineups_list, ignore_index=True)
print(f"Done. Total lineup entries: {len(lineups_raw):,}")

---
## Section 4 - Pull 360 Freeze-Frame Data

360 data contains the locations of all visible players on the pitch at the moment of specific events. Not available for every match, but we try each one.

In [ ]:
frames_list = []
matches_with_360 = 0
matches_without_360 = 0

print("Pulling 360 frame data...\n")
for idx, row in inventory.iterrows():
    match_id = int(row['match_id'])
    tourney  = row['tournament']
    opponent = row['opponent']
    
    try:
        frames = sb.frames(match_id=match_id)
        if frames is not None and not frames.empty:
            frames = frames.copy()
            frames['match_id']   = match_id
            frames['tournament'] = tourney
            frames_list.append(frames)
            matches_with_360 += 1
            print(f"  [OK]  Match {match_id} ({tourney} vs {opponent}) -- {len(frames)} rows")
        else:
            matches_without_360 += 1
            print(f"  [--]  Match {match_id} ({tourney} vs {opponent}) -- No 360 data")
    except Exception as e:
        matches_without_360 += 1
        print(f"  [ERR] Match {match_id} ({tourney} vs {opponent}) -- {e}")
        
    time.sleep(1.5)  # Throttle to avoid 429 Rate Limit

if frames_list:
    frames_raw = pd.concat(frames_list, ignore_index=True)
    print(f"\nDone. 360 data for {matches_with_360} matches. Total rows: {len(frames_raw):,}")
else:
    frames_raw = pd.DataFrame()
    print(f"\nDone. No 360 data was found for ANY match.")

---
## Section 5 - Save Raw Datasets

In [ ]:
def prep_for_parquet(df):
    """Cast list/dict columns to strings so parquet can store them."""
    df_out = df.copy()
    for col in df_out.columns:
        if df_out[col].apply(lambda x: isinstance(x, (list, dict))).any():
            df_out[col] = df_out[col].astype(str)
    return df_out

os.makedirs(OUTPUTS_DATA_DIR, exist_ok=True)

events_raw_pq = prep_for_parquet(events_raw)
events_raw_pq.to_parquet(os.path.join(OUTPUTS_DATA_DIR, 'events_raw.parquet'), index=False)
print(f"Saved events_raw.parquet ({len(events_raw):,} rows)")

lineups_raw_pq = prep_for_parquet(lineups_raw)
lineups_raw_pq.to_parquet(os.path.join(OUTPUTS_DATA_DIR, 'lineups_raw.parquet'), index=False)
print(f"Saved lineups_raw.parquet ({len(lineups_raw):,} rows)")

if not frames_raw.empty:
    frames_raw_pq = prep_for_parquet(frames_raw)
    frames_raw_pq.to_parquet(os.path.join(OUTPUTS_DATA_DIR, 'frames_360_raw.parquet'), index=False)
    print(f"Saved frames_360_raw.parquet ({len(frames_raw):,} rows)")
else:
    print("No 360 data to save.")

print("\nAll raw datasets saved.")

---
## Section 6 - Summary

In [ ]:
print("=" * 60)
print("  RAW DATA INVENTORY SUMMARY")
print("=" * 60)
print(f"  Total Events     : {len(events_raw):,}")
print(f"  Total Lineup Rows: {len(lineups_raw):,}")
print(f"  Total 360 Rows   : {len(frames_raw):,}")
print("-" * 60)
print("\nEvents by Tournament:")
print(events_raw['tournament'].value_counts().to_string())
print("\nEvents by Match (Spain's perspective):")
print(events_raw.groupby(['tournament', 'opponent'])['id'].count().to_string())
print("=" * 60)
print("\n  NEXT: Notebook 03 -- Data Validation & Cleaning")
print("=" * 60)